# Prosta-AI: entrenamiento y evaluación en Google Colab

Aplicación académica y experimental, **no diagnóstica**. Este cuaderno ajusta una UNet 3D (MONAI) con los casos de
entrenamiento de MSD Task05_Prostate, la evalúa sobre el conjunto de prueba y permite descargar los pesos.

Antes de empezar: menú *Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU*.

## 1. Comprobar la GPU

In [ ]:
!nvidia-smi
import torch
print('CUDA disponible:', torch.cuda.is_available())

## 2. Clonar el repositorio

Cambie `URL_REPO` si trabaja con un fork. La rama puede ser `main` o `desarrollo`.

In [ ]:
URL_REPO = 'https://github.com/yimis-source/ProstaAIReal.git'
RAMA = 'main'
import os
if not os.path.isdir('/content/ProstaAIReal'):
    !git clone --branch {RAMA} {URL_REPO} /content/ProstaAIReal
%cd /content/ProstaAIReal
!git log -1 --oneline

## 3. Instalar dependencias con versiones exactas (RA-01)

Son las mismas versiones que fija `backend/pyproject.toml`. Si Colab pide reiniciar el entorno tras instalar,
reinícielo y vuelva a ejecutar desde la celda 2.

In [ ]:
!pip install -q numpy==1.26.4 nibabel==5.3.2 SimpleITK==2.4.0 monai==1.4.0 torch==2.5.1
import sys
sys.path.insert(0, '/content/ProstaAIReal/backend')  # para importar prosta_ai.pipeline sin instalar el backend
sys.path.insert(0, '/content/ProstaAIReal')          # para importar entrenamiento.*
import os
os.environ['PYTHONPATH'] = '/content/ProstaAIReal/backend:/content/ProstaAIReal'  # lo heredan los comandos con !python

## 4. Descargar los datos públicos (MSD Task05_Prostate, CC-BY-SA 4.0)

Usa el script del ciclo 1, que verifica la descarga y deja los datos en `datos/Task05_Prostate` (no se suben al
repositorio). La ficha con la fuente, la licencia y la cita está en `docs/datos/fichas/` (RC-03).

In [ ]:
!python scripts/descargar_datos.py --destino datos
!ls datos/Task05_Prostate/imagesTr | head -5

## 5. Descargar el modelo base

El ajuste fino parte de los pesos del bundle `prostate_mri_anatomy` (Apache 2.0).

In [ ]:
!python scripts/descargar_modelo.py --destino modelos

## 6. División por paciente 70/15/15 (RC-04)

Semilla fija 2026. La división ya está versionada en `entrenamiento/division_task05.json` (ciclo 2); aquí solo se
verifica que ningún paciente aparece en dos conjuntos y que la huella coincide.

In [ ]:
!python -c "from entrenamiento.division import cargar_division; d = cargar_division('entrenamiento/division_task05.json'); print(d['cantidades'], d['huella_sha256'][:16])"

## 7. Entrenamiento (ajuste fino)

Usa exactamente las mismas transformaciones de preparación que la inferencia (RF-09). Cambie `MAX_EPOCAS` según el
tiempo disponible: con la GPU gratuita de Colab, unas 100 épocas son un tiempo razonable. Para una comprobación
rápida use `!python -m entrenamiento.entrenar --prueba-rapida`.

In [ ]:
MAX_EPOCAS = 100
NOMBRE = 'prostate_task05_ft'
!python -m entrenamiento.entrenar --max-epocas {MAX_EPOCAS} --dispositivo cuda --nombre {NOMBRE}

## 8. Evaluación sobre el conjunto de prueba (RF-23, RF-24)

Se usa la misma ruta de inferencia que usará la aplicación (`build_pipeline`). El informe queda en `reports/evaluacion/`.
Se evalúa el modelo ajustado y, como referencia, el modelo base.

In [ ]:
!python -m entrenamiento.evaluar --modelo {NOMBRE} --dispositivo cuda
!python -m entrenamiento.evaluar --modelo prostate_mri_anatomy --dispositivo cuda
!ls reports/evaluacion

In [ ]:
import glob
informe = sorted(glob.glob(f'reports/evaluacion/evaluacion_{NOMBRE}_*.md'))[-1]
print(open(informe, encoding='utf-8').read())

## 9. Descargar los pesos y los informes

Se comprime la carpeta del modelo (pesos, `preparacion.json`, metadatos, división) y los informes.
Para usarlos, descomprima en `modelos/<nombre>/` del repositorio y evalúe con `entrenamiento/evaluar.py`.

In [ ]:
!zip -r -q resultados_prosta_ai.zip modelos/{NOMBRE} reports/evaluacion entrenamiento/division_task05.json
from google.colab import files
files.download('resultados_prosta_ai.zip')

## Limitaciones

- Con 32 casos el conjunto de prueba tiene solo 5; las métricas son muy variables.
- Task05 es un solo conjunto público; no hay validación externa ni clínica.
- Consulte la sección de limitaciones del informe generado (posible sesgo por solapamiento de datos del modelo base).
- Uso académico: no apto para diagnóstico.